In [ ]:
# Setup: installs MAX and this book's `minillm` package with uv if they are
# missing, as on Colab or Kaggle (turn on Internet in Kaggle's settings
# first). In the book's pixi environment, this cell does nothing.
import importlib.util
import os
import shutil
import subprocess
import sys

BOOK_REF = "main"  # Branch or tag of github.com/modular/max-llm-book.
BOOK_SRC = os.environ.get(
    "MAX_LLM_BOOK_SRC",
    f"git+https://github.com/modular/max-llm-book@{BOOK_REF}",
)


def uv_install(*packages: str) -> None:
    if shutil.which("uv") is None:
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q", "uv"], check=True
        )
    subprocess.run(
        [
            "uv",
            "pip",
            "install",
            "--python",
            sys.executable,
            "-q",
            "--prerelease",
            "allow",
            "--index-strategy",
            "unsafe-best-match",
            "--extra-index-url",
            "https://whl.modular.com/nightly/simple/",
            *packages,
        ],
        check=True,
    )


if importlib.util.find_spec("max") is None:
    uv_install("max[all]")
if importlib.util.find_spec("minillm") is None:
    uv_install(BOOK_SRC)
# Cells run `max` and `mojo` as commands; find them next to this Python even
# when the environment isn't activated.
os.environ["PATH"] = (
    os.path.dirname(sys.executable) + os.pathsep + os.environ["PATH"]
)
# MAX's matmul needs an NVIDIA GPU from the Ampere generation (compute
# capability 8.0) or newer. Hide older ones, such as Colab's T4, so MAX uses
# the CPU instead of failing.
if shutil.which("nvidia-smi"):
    caps = subprocess.run(
        ["nvidia-smi", "--query-gpu=compute_cap", "--format=csv,noheader"],
        capture_output=True,
        text=True,
    ).stdout.split()
    usable = [str(i) for i, cap in enumerate(caps) if float(cap) >= 8.0]
    if len(usable) < len(caps):
        os.environ["CUDA_VISIBLE_DEVICES"] = ",".join(usable)
        print(
            f"Using {len(usable)} of {len(caps)} GPUs; MAX needs compute capability 8.0 or newer."
        )

# 7.4 Modular's cloud

Everything in this book ran on your own machine: models of up to 0.6B
parameters, with multi-device meshes simulated on a CPU. Serving
state-of-the-art open models to real traffic takes more: the largest models span
many GPUs, capacity has to follow demand, and someone has to respond when a node
fails. [Modular's cloud](https://www.modular.com/) is an inference service that
runs those models on MAX, the framework you used in this book. This last section
points your client code at it by changing the base URL, the API key, and the
model name.

In [ ]:
import os

from openai import OpenAI

from minillm.config import MODEL_ID
from minillm.server import start_max_serve

## Two ways to call a hosted model

Modular's cloud serves popular open models behind the same OpenAI-compatible
API that the earlier sections of this chapter used:

- [Shared endpoints](https://www.modular.com/inference/shared-endpoints)
  are billed per token, and their capacity scales automatically, down to
  zero when idle.
- [Dedicated endpoints](https://www.modular.com/inference/dedicated-endpoints)
  reserve GPUs for you and are billed per minute, for consistent latency and
  guaranteed throughput.

The [pricing](https://www.modular.com/pricing) page compares them.

## The same client, a different URL

Every server in [Chapter 7](https://llm.modular.com/ch07.html) spoke the OpenAI
API, and so do Modular's endpoints. Here is the client from
[Section 7.1](https://llm.modular.com/07_01_serve_your_model.html) against a
local server again:

In [ ]:
server = start_max_serve(MODEL_ID, "--quantization-encoding", "float32")


def ask(client: OpenAI, model: str, question: str) -> str:
    completion = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": question}],
        max_tokens=60,
        temperature=0,
    )
    return completion.choices[0].message.content


local = OpenAI(base_url="http://localhost:8000/v1", api_key="unused")
print(ask(local, MODEL_ID, "What is the capital of France?"))
server.terminate()
_ = server.wait()

To call a hosted model instead:

1. Sign up at [console.modular.com](https://console.modular.com/signup).
2. Create an API key with model access on the
   [API keys](https://console.modular.com/api_keys) page, and set it as the
   `MODULAR_API_KEY` environment variable. In Colab, add it as a secret and
   copy it into `os.environ`.
3. Pick a model on the [endpoints](https://console.modular.com/endpoints)
   page, and set its model ID as `MODULAR_MODEL`.

Only the base URL, the key, and the model name change. The cell skips the
request when no key is set:

In [ ]:
api_key = os.environ.get("MODULAR_API_KEY")
if api_key is None:
    print("Set MODULAR_API_KEY and MODULAR_MODEL to call a hosted model.")
else:
    hosted = OpenAI(base_url="https://api.modular.com/v1", api_key=api_key)
    print(
        ask(
            hosted,
            os.environ["MODULAR_MODEL"],
            "What is the capital of France?",
        )
    )

The API is the same, so code written against it, such as the structured output
in [Section 7.3](https://llm.modular.com/07_03_structured_output.html), works
with hosted models that support the feature. `max benchmark` measures a hosted
endpoint as it measured your local server. The `mcloud` backend sends the key
from `OPENAI_API_KEY`. A hosted endpoint manages its own cache and doesn't
expose server metrics, so the command skips both. If the model ID isn't a
Hugging Face repo, add `--tokenizer` with the repo of its tokenizer:

```bash
OPENAI_API_KEY=$MODULAR_API_KEY max benchmark --backend mcloud \
  --base-url https://api.modular.com --endpoint /v1/chat/completions \
  --no-flush-prefix-cache --no-collect-server-stats \
  --model $MODULAR_MODEL \
  --dataset-name random --random-input-len 128 --random-output-len 32 \
  --num-prompts 32 --max-concurrency 8
```

Before you choose between shared and dedicated endpoints, benchmark with your
real prompt and answer lengths; the latency targets from
[Section 7.2](https://llm.modular.com/07_02_benchmark_serving.html) apply
unchanged. The handbook's
[choosing the right model](https://handbook.modular.com/getting-started/choosing-the-right-model/)
and
[calculating GPU memory](https://handbook.modular.com/getting-started/calculating-gpu-memory-for-llms/)
pages help you choose a model and estimate the GPUs it needs.

## What you built

You started this book with three lines of `AutoPipeline`. Since then you:

- wrote SmolLM2 from tensors up, checked it against MAX's implementation, and
  added sampling and a KV cache
  ([Chapter 1](https://llm.modular.com/ch01.html));
- sharded it across a device mesh
  ([Chapter 2](https://llm.modular.com/ch02.html));
- wrote a Mojo kernel and built a graph by hand
  ([Chapter 3](https://llm.modular.com/ch03.html));
- tested, benchmarked, and profiled it
  ([Chapter 4](https://llm.modular.com/ch04.html));
- built a serving engine with continuous batching, a paged KV cache, prefix
  caching, and speculative decoding
  ([Chapter 5](https://llm.modular.com/ch05.html));
- brought up Qwen3, Gemma 3, and a mixture of experts
  ([Chapter 6](https://llm.modular.com/ch06.html));
- served your own architecture, measured it, and constrained its output
  ([Chapter 7](https://llm.modular.com/ch07.html)).

A model takes the same path into MAX, at a larger size. To keep going:

- Read MAX's guide to
  [model bring-up](https://max.modular.com/develop/model-bringup-workflow) and
  port a model you care about.
- Work through the
  [Mojo GPU puzzles](https://github.com/modular/mojo-gpu-puzzles) to write the
  kernels behind it.
- Read the [handbook](https://handbook.modular.com/) for the ideas behind every
  chapter.
- [Sign up](https://console.modular.com/signup) to run state-of-the-art open
  models on Modular's cloud, or
  [request a demo](https://www.modular.com/request-demo) to talk about your
  workload.